# Optimizer Selection via PPO Contextual Bandit

This notebook trains an agent to pick the **single best optimizer** (out of 5
options) for a given quantum circuit, based on the circuit's pre-optimization
features.

**Framing:** this is a *contextual bandit* -- one decision per circuit, full
episode terminates after that decision. PPO is used because it tends to be
more stable than DQN on continuous (`Box`) observation spaces and doesn't
require a replay buffer / target network, which add unnecessary complexity for
a single-step task.

**Key fixes vs. the original version:**
- Train/val/test split is done on **unique `original_circuit` values**, so all
  5 optimizer outcomes for a given circuit stay together in one split (no leakage).
- The PPO model is trained on `train_df` (previously it was accidentally
  trained on `test_df`).
- A held-out **validation set** is used to monitor generalization during
  training without touching the test set.
- Observations are **always the original (pre-optimization) circuit
  features**, consistently across training and evaluation.
- A custom callback tracks mean episode reward over training for a
  convergence plot.
- All baselines (random, always-best-single-optimizer, oracle) are computed
  per split for apples-to-apples comparison.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

pd.set_option('display.max_columns', None)


## 2. Load data and one-hot encode circuit categories

In [ ]:
master_df = pd.read_csv(
    "./trajectories_db_analysis/master_df.csv"
)

master_df = pd.get_dummies(
    master_df,
    columns=["original_category"],
    drop_first=False,
    dtype=int,
)

master_df.head()


## 3. Train / validation / test split

Split on **unique `original_circuit` values** (not on `master_df` rows
directly). Each circuit has 5 rows in `master_df` -- one per optimizer -- and
all 5 must end up in the same split, otherwise the agent could indirectly see
test-circuit outcomes during training.


In [ ]:
from sklearn.model_selection import train_test_split

circuits = master_df["original_circuit"].unique()

# First split off the test set (10%)
train_val_circuits, test_circuits = train_test_split(
    circuits,
    test_size=0.10,
    random_state=42,
)

# Then split the remainder into train (80% of total) and val (10% of total)
# val_size relative to train_val_circuits: 0.10 / 0.90 ~= 0.1111
train_circuits, val_circuits = train_test_split(
    train_val_circuits,
    test_size=0.10 / 0.90,
    random_state=42,
)

train_df = master_df[master_df["original_circuit"].isin(train_circuits)].copy()
val_df = master_df[master_df["original_circuit"].isin(val_circuits)].copy()
test_df = master_df[master_df["original_circuit"].isin(test_circuits)].copy()

print(f"Train circuits: {len(train_circuits)}  ({len(train_df)} rows)")
print(f"Val circuits:   {len(val_circuits)}  ({len(val_df)} rows)")
print(f"Test circuits:  {len(test_circuits)}  ({len(test_df)} rows)")

# Sanity check: no circuit overlap between splits
assert set(train_circuits).isdisjoint(val_circuits)
assert set(train_circuits).isdisjoint(test_circuits)
assert set(val_circuits).isdisjoint(test_circuits)
print("No circuit overlap between splits - OK")


## 4. Environment

Same core idea as the original `OptimizerEnv`, with two changes:

1. `reset()` cycles through circuits without replacement each "epoch" (instead
   of pure random sampling every reset), so PPO sees the full training set
   evenly over `total_timesteps`.
2. The observation returned is **always** the row's *original* (pre-optimization)
   feature values -- the agent should choose an optimizer based on what the
   circuit looks like *before* any optimization, never based on an
   already-optimized version.


In [ ]:
class OptimizerEnv(gym.Env):
    """Contextual-bandit environment: pick one of 5 optimizers for a circuit.

    Each episode = exactly one decision (single-step). The observation is the
    *pre-optimization* feature vector for a circuit; the reward is the
    fractional reduction in 2-qubit gate count achieved by the chosen
    optimizer.
    """

    def __init__(self, df, feature_columns, shuffle_each_epoch=True):
        super().__init__()

        self.df = df
        self.feature_columns = feature_columns
        self.shuffle_each_epoch = shuffle_each_epoch

        self.observation_space = gym.spaces.Box(
            low=0,
            high=np.inf,
            shape=(len(feature_columns),),
            dtype=np.float32,
        )

        # 5 possible optimization strategies
        self.action_space = gym.spaces.Discrete(5)

        self._action_to_optimizer = {
            0: "qiskit_standard",
            1: "qiskit_ai",
            2: "tket",
            3: "wisq_rules",
            4: "wisq_bqskit",
        }

        # Precompute, per circuit, the row used for observations: the first
        # row for that circuit (feature columns are identical across the 5
        # optimizer rows for a given circuit, since they describe the
        # *original* circuit, not the optimized output).
        self._circuits = self.df["original_circuit"].unique()
        self._circuit_to_obs_row = {
            c: self.df[self.df["original_circuit"] == c].iloc[0]
            for c in self._circuits
        }

        self._epoch_order = None
        self._epoch_idx = 0

    def _next_circuit(self):
        if self._epoch_order is None or self._epoch_idx >= len(self._epoch_order):
            self._epoch_order = self._circuits.copy()
            if self.shuffle_each_epoch:
                self.np_random.shuffle(self._epoch_order)
            self._epoch_idx = 0

        circuit = self._epoch_order[self._epoch_idx]
        self._epoch_idx += 1
        return circuit

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        circuit = self._next_circuit()
        self.current_circuit = circuit
        self.current_row = self._circuit_to_obs_row[circuit]

        self.state = self.current_row[self.feature_columns].to_numpy(dtype=np.float32)
        return self.state, {}

    def step(self, action):
        action = int(action)
        optimizer = self._action_to_optimizer[action]

        mask = (
            (self.df["original_circuit"] == self.current_circuit)
            & (self.df["opt_1"] == optimizer)
        )
        matches = self.df.loc[mask]

        if matches.empty:
            raise ValueError(
                f"No row found for circuit={self.current_circuit} "
                f"and optimizer={optimizer}"
            )

        row = matches.iloc[0]
        old_2q = row["initial_two_qubit_gates"]
        new_2q = row["final_two_qubit_gates"]

        reward = 0.0 if old_2q == 0 else (old_2q - new_2q) / old_2q

        # Single-step episode: terminate immediately. The next observation is
        # irrelevant for the bandit's bootstrap target (gamma is effectively
        # unused), but SB3 still requires a valid observation -- return the
        # same pre-optimization features.
        terminated = True
        truncated = False
        info = {"chosen_optimizer": optimizer, "circuit": self.current_circuit}

        return self.state, reward, terminated, truncated, info


## 5. Feature columns and environment instances

In [ ]:
feature_columns = [
    "initial_num_qubits",
    "initial_depth",
    "initial_two_qubit_gates",
    "initial_total_gates",
    "input_two_qubit_depth",
    "original_category_clifford",
    "original_category_feynman",
    "original_category_guoq_ibmnew",
    "original_category_misc",
    "original_category_qaoa",
    "original_category_qft",
    "original_category_qv",
    "original_category_square-heisenberg",
]

# Training env (the model is trained on this -- and ONLY this)
train_env_raw = OptimizerEnv(train_df, feature_columns)
check_env(train_env_raw)

# Wrap with Monitor so episode rewards are logged for the callback below
train_env = Monitor(train_env_raw)

# Separate envs for evaluation (val/test). shuffle_each_epoch=False isn't
# strictly necessary since we'll evaluate circuit-by-circuit manually, but
# kept for clarity / potential future use with SB3's evaluate_policy.
val_env = OptimizerEnv(val_df, feature_columns, shuffle_each_epoch=False)
test_env = OptimizerEnv(test_df, feature_columns, shuffle_each_epoch=False)


## 6. Train PPO

`gamma` is left at a low value since this is a single-step bandit (no
meaningful future to discount) -- it has negligible effect here but is set
explicitly for clarity. A callback records the rolling mean episode reward so
we can plot convergence.


In [ ]:
class RewardHistoryCallback(BaseCallback):
    """Records mean episode reward each time Monitor logs a completed episode."""

    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []

    def _on_step(self) -> bool:
        # Monitor wrapper stores completed-episode info in `infos`
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
        return True


model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=3e-4,
    n_steps=256,        # rollout buffer size before each PPO update
    batch_size=64,
    gamma=0.0,          # single-step episodes: no future to discount
    verbose=1,
    seed=42,
)

reward_callback = RewardHistoryCallback()
model.learn(total_timesteps=20_000, callback=reward_callback)

model.save("optimizer_ppo")


## 7. Training reward curve

In [ ]:
rewards = np.array(reward_callback.episode_rewards)

if len(rewards) > 0:
    window = 50
    if len(rewards) >= window:
        rolling_mean = np.convolve(rewards, np.ones(window) / window, mode="valid")
    else:
        rolling_mean = rewards

    plt.figure(figsize=(8, 4))
    plt.plot(rewards, alpha=0.3, label="episode reward")
    plt.plot(
        np.arange(len(rolling_mean)) + (len(rewards) - len(rolling_mean)),
        rolling_mean,
        label=f"rolling mean ({window})",
    )
    plt.xlabel("Episode")
    plt.ylabel("Reward (2Q gate reduction fraction)")
    plt.title("PPO training reward")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No completed episodes recorded.")


## 8. Evaluation helpers

`evaluate_policy_on_df` mirrors the environment's logic: for each unique
circuit in a split, build the *pre-optimization* feature vector, get the
model's action, look up the actual outcome for that (circuit, optimizer)
pair, and compute the same reward used in training.

The baseline functions reuse the same reward definition for direct
comparison.


In [ ]:
action_to_optimizer = {
    0: "qiskit_standard",
    1: "qiskit_ai",
    2: "tket",
    3: "wisq_rules",
    4: "wisq_bqskit",
}


def _row_reward(row):
    old_2q = row["initial_two_qubit_gates"]
    new_2q = row["final_two_qubit_gates"]
    return 0.0 if old_2q == 0 else (old_2q - new_2q) / old_2q


def evaluate_policy_on_df(model, df, feature_columns, deterministic=True):
    rewards = []

    for circuit in df["original_circuit"].unique():
        circuit_rows = df[df["original_circuit"] == circuit]

        # Pre-optimization features: identical across the 5 optimizer rows,
        # so the first row works fine.
        obs_row = circuit_rows.iloc[0]
        obs = obs_row[feature_columns].to_numpy(dtype=np.float32)

        action, _ = model.predict(obs, deterministic=deterministic)
        optimizer = action_to_optimizer[int(action)]

        chosen_rows = circuit_rows[circuit_rows["opt_1"] == optimizer]
        if chosen_rows.empty:
            continue  # shouldn't happen if data is complete

        rewards.append(_row_reward(chosen_rows.iloc[0]))

    return np.mean(rewards), rewards


def random_baseline(df, optimizer_names, seed=0):
    rng = np.random.default_rng(seed)
    rewards = []

    for circuit in df["original_circuit"].unique():
        circuit_rows = df[df["original_circuit"] == circuit]
        chosen_optimizer = rng.choice(optimizer_names)

        chosen_rows = circuit_rows[circuit_rows["opt_1"] == chosen_optimizer]
        if chosen_rows.empty:
            continue

        rewards.append(_row_reward(chosen_rows.iloc[0]))

    return np.mean(rewards)


def fixed_optimizer_baseline(df, optimizer_name):
    rewards = []

    for circuit in df["original_circuit"].unique():
        circuit_rows = df[df["original_circuit"] == circuit]
        chosen_rows = circuit_rows[circuit_rows["opt_1"] == optimizer_name]
        if chosen_rows.empty:
            continue

        rewards.append(_row_reward(chosen_rows.iloc[0]))

    return np.mean(rewards)


def oracle_reward(df):
    rewards = []

    for circuit in df["original_circuit"].unique():
        rows = df[df["original_circuit"] == circuit]
        r = (
            (rows["initial_two_qubit_gates"] - rows["final_two_qubit_gates"])
            / rows["initial_two_qubit_gates"]
        )
        rewards.append(r.max())

    return np.mean(rewards)


## 9. Results: PPO agent vs. baselines, per split

In [ ]:
optimizer_names = list(action_to_optimizer.values())

results = {}

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    ppo_reward, _ = evaluate_policy_on_df(model, df, feature_columns)
    rand_reward = random_baseline(df, optimizer_names, seed=42)
    oracle_r = oracle_reward(df)

    results[split_name] = {
        "PPO agent": ppo_reward,
        "Random": rand_reward,
        "Oracle (best possible)": oracle_r,
    }

# Add "always pick X" baselines for each optimizer, computed on test only
for opt in optimizer_names:
    results.setdefault("test_fixed_baselines", {})[f"Always {opt}"] = (
        fixed_optimizer_baseline(test_df, opt)
    )

results_df = pd.DataFrame({
    k: v for k, v in results.items() if k != "test_fixed_baselines"
}).T

print(results_df.round(4))
print()
print("Fixed-optimizer baselines (test set):")
for k, v in results["test_fixed_baselines"].items():
    print(f"  {k}: {v:.4f}")


## 10. Action distribution

Sanity check: which optimizers does the agent actually choose, and how does
that compare to which optimizer is *truly* best per circuit (oracle choice)?
A policy that collapses to always picking one action is a red flag (it likely
means that optimizer dominates on average, or training hasn't converged).


In [ ]:
def policy_action_distribution(model, df, feature_columns):
    counts = {name: 0 for name in action_to_optimizer.values()}

    for circuit in df["original_circuit"].unique():
        circuit_rows = df[df["original_circuit"] == circuit]
        obs_row = circuit_rows.iloc[0]
        obs = obs_row[feature_columns].to_numpy(dtype=np.float32)

        action, _ = model.predict(obs, deterministic=True)
        counts[action_to_optimizer[int(action)]] += 1

    return counts


def oracle_action_distribution(df):
    counts = {name: 0 for name in action_to_optimizer.values()}

    for circuit in df["original_circuit"].unique():
        rows = df[df["original_circuit"] == circuit]
        rewards = (
            (rows["initial_two_qubit_gates"] - rows["final_two_qubit_gates"])
            / rows["initial_two_qubit_gates"]
        )
        best_idx = rewards.idxmax()
        best_optimizer = rows.loc[best_idx, "opt_1"]
        counts[best_optimizer] += 1

    return counts


for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"--- {split_name} ---")
    print("Agent choices:", policy_action_distribution(model, df, feature_columns))
    print("Oracle choices:", oracle_action_distribution(df))
    print()
